# Lab 6 — Retrieval-Augmented Generation (RAG)


**Corpus:** 6 Wikipedia articles on generative AI topics (`docs/`): Transformer, Large language model,
Attention mechanism, Diffusion model, Generative adversarial network, Retrieval-augmented generation.

The pipeline implemented in this notebook:

```
 docs/*.md ──► load ──► chunk ──► embed (sentence-transformers) ──► ChromaDB (vector store)
                                                                          │
 question ──► embed ──► similarity search (top-k chunks) ◄────────────────┘
                              │
                              ▼
                 prompt = system + [context chunks] + question
                              │
                              ▼
                   Claude (Anthropic API) ──► grounded answer + sources
```

Steps:
1. Setup and configuration
2. Load documents
3. Chunk documents
4. Embed chunks
5. Store in a vector database (ChromaDB)
6. Retrieve relevant chunks for a question
7. Generate an answer with an LLM (Claude)
8. Evaluate: retrieval hit-rate, RAG vs. no-RAG comparison

## 1. Setup and configuration

Install the dependencies once (uncomment the line below or run `pip install -r requirements.txt` in the terminal), then put your Anthropic key in a `.env` file next to this notebook (see `.env.example`).

In [1]:
# %pip install -q -r requirements.txt

In [2]:
import os, re, json, textwrap, time
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()  # reads .env -> ANTHROPIC_API_KEY, CLAUDE_MODEL

DOCS_DIR = Path("docs")
DB_DIR = Path("chroma_db")           # ChromaDB persists here between runs
COLLECTION_NAME = "lab6_genai_wiki"

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # small, fast, 384-dim, runs on CPU
CHUNK_SIZE = 900        # characters per chunk (~200 tokens)
CHUNK_OVERLAP = 150     # characters shared between neighbouring chunks
TOP_K = 5               # chunks passed to the LLM

CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-opus-5-5")

assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY is missing - create a .env file (see .env.example)"
print("Model:", CLAUDE_MODEL)
print("Docs:", sorted(p.name for p in DOCS_DIR.glob("*.md")))

Model: claude-opus-5-5
Docs: ['attention_mechanism.md', 'diffusion_model.md', 'generative_adversarial_network.md', 'large_language_model.md', 'retrieval_augmented_generation.md', 'transformer.md']


## 2. Load documents

Each file is a cleaned Wikipedia article in Markdown. We keep the file name as `source` and the first `# heading` as `title`; both become metadata that travels with every chunk so answers can cite where they came from.

In [3]:
def load_documents(docs_dir: Path) -> list[dict]:
    documents = []
    for path in sorted(docs_dir.glob("*.md")):
        text = path.read_text(encoding="utf-8")
        text = re.sub(r"<!--.*?-->\s*", "", text, flags=re.S)   # drop the HTML source comment
        title_match = re.search(r"^# (.+)$", text, flags=re.M)
        documents.append({
            "source": path.name,
            "title": title_match.group(1).strip() if title_match else path.stem,
            "text": text.strip(),
        })
    return documents

documents = load_documents(DOCS_DIR)

print(f"{'source':40} {'title':36} {'words':>7} {'~pages':>7}")
for d in documents:
    words = len(d["text"].split())
    print(f"{d['source']:40} {d['title'][:36]:36} {words:7} {words/500:7.1f}")
print("\nTotal words:", sum(len(d["text"].split()) for d in documents))

source                                   title                                  words  ~pages
attention_mechanism.md                   Attention                               3128     6.3
diffusion_model.md                       Diffusion model                         6400    12.8
generative_adversarial_network.md        Generative adversarial network          6335    12.7
large_language_model.md                  Large language model                    7864    15.7
retrieval_augmented_generation.md        Retrieval-augmented generation          1669     3.3
transformer.md                           Transformer                             8613    17.2

Total words: 34009


## 3. Chunk documents

LLMs have a context limit and embeddings work best on focused passages, so each article is split into chunks.
The strategy is **structure-aware**:

1. split the article on Markdown headings (`##`, `###` …) so a chunk never mixes two sections;
2. split long sections further on paragraph boundaries, falling back to sentences, until each piece is at most `CHUNK_SIZE` characters;
3. prepend `CHUNK_OVERLAP` characters from the previous chunk so that facts split across a boundary still appear whole in one of them.

Every chunk carries metadata: `source`, `title`, `section`, `chunk_id`.

In [4]:
HEADING_RE = re.compile(r"^(#{1,6})\s+(.*)$", flags=re.M)

def split_into_sections(text: str) -> list[tuple[str, str]]:
    # returns [(section_title, section_text), ...]
    matches = list(HEADING_RE.finditer(text))
    if not matches:
        return [("", text)]
    sections = []
    for i, m in enumerate(matches):
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        body = text[start:end].strip()
        if body:
            sections.append((m.group(2).strip(), body))
    return sections

def split_long_text(text: str, max_len: int) -> list[str]:
    # recursive split: paragraphs -> lines -> sentences -> hard cut
    text = text.strip()
    if len(text) <= max_len:
        return [text] if text else []
    for sep in ("\n\n", "\n", ". "):
        if sep in text:
            parts = text.split(sep)
            break
    else:  # no separator left: hard cut
        return [text[i:i + max_len] for i in range(0, len(text), max_len)]
    pieces, buf = [], ""
    for part in parts:
        part = part.strip()
        if not part:
            continue
        if len(part) > max_len:          # a single paragraph/sentence that is too long
            if buf:
                pieces.append(buf)
                buf = ""
            pieces.extend(split_long_text(part, max_len))
            continue
        candidate = f"{buf}{sep}{part}" if buf else part
        if len(candidate) <= max_len:
            buf = candidate
        else:
            pieces.append(buf)
            buf = part
    if buf:
        pieces.append(buf)
    return pieces

def chunk_documents(documents: list[dict], chunk_size: int, overlap: int) -> list[dict]:
    chunks = []
    for doc in documents:
        prev_tail = ""
        for section_title, section_text in split_into_sections(doc["text"]):
            for piece in split_long_text(section_text, chunk_size):
                content = (prev_tail + " " + piece).strip() if prev_tail else piece
                chunks.append({
                    "id": f"{doc['source']}::{len(chunks)}",
                    "text": content,
                    "metadata": {
                        "source": doc["source"],
                        "title": doc["title"],
                        "section": section_title or doc["title"],
                        "chunk_id": len(chunks),
                    },
                })
                prev_tail = piece[-overlap:] if overlap else ""
    return chunks

chunks = chunk_documents(documents, CHUNK_SIZE, CHUNK_OVERLAP)
lengths = [len(c["text"]) for c in chunks]
print(f"{len(chunks)} chunks | min {min(lengths)} / avg {sum(lengths)//len(lengths)} / max {max(lengths)} characters")
print("\nChunks per document:")
for d in documents:
    print(f"  {d['source']:40} {sum(c['metadata']['source']==d['source'] for c in chunks):4}")

print("\nExample chunk:")
example = chunks[len(chunks)//2]
print(json.dumps(example["metadata"], indent=2))
print(textwrap.fill(example["text"][:600], 100), "...")

405 chunks | min 158 / avg 698 / max 1051 characters

Chunks per document:
  attention_mechanism.md                     37
  diffusion_model.md                         74
  generative_adversarial_network.md          74
  large_language_model.md                    96
  retrieval_augmented_generation.md          21
  transformer.md                            103

Example chunk:
{
  "source": "large_language_model.md",
  "title": "Large language model",
  "section": "Cost",
  "chunk_id": 202
}
trained in different ways. In particular, GPT models are first pretrained to predict the next word
on a large amount of data, before being fine-tuned. Substantial infrastructure is necessary for
training the largest models. The tendency towards larger models is visible in the list of large
language models. For example, the training of GPT-2 (i.e. a 1.5-billion-parameter model) in 2019
cost $50,000, while training of the PaLM (i.e. a 540-billion-parameter model) in 2022 cost $8
million, and Megatron-

## 4. Embed chunks

An **embedding model** maps text to a dense vector such that semantically similar texts are close
(by cosine similarity). We use `all-MiniLM-L6-v2` from `sentence-transformers`: 384 dimensions, ~80 MB,
fast on CPU. The model is downloaded from Hugging Face the first time it is used.

The *same* model must be used for the documents and, later, for the questions.

In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer(EMBED_MODEL_NAME)

def embed_texts(texts: list[str]) -> np.ndarray:
    # normalize_embeddings=True -> dot product == cosine similarity
    return embedder.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=False)

# quick sanity check: similar sentences should score higher than unrelated ones
a, b, c = embed_texts([
    "Self-attention lets every token look at every other token.",
    "The attention mechanism relates different positions of a sequence.",
    "GANs train a generator against a discriminator.",
])
print("dimension:", a.shape[0])
print("sim(attention, attention-paraphrase) =", round(float(a @ b), 3))
print("sim(attention, GAN)                  =", round(float(a @ c), 3))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

dimension: 384
sim(attention, attention-paraphrase) = 0.456
sim(attention, GAN)                  = 0.116


In [6]:
t0 = time.time()
chunk_embeddings = embed_texts([c["text"] for c in chunks])
print(f"Embedded {len(chunks)} chunks -> matrix {chunk_embeddings.shape} in {time.time()-t0:.1f}s")

Embedded 405 chunks -> matrix (405, 384) in 15.2s


## 5. Store in a vector database (ChromaDB)

ChromaDB stores the vectors together with the chunk text and metadata and answers nearest-neighbour queries.
We use a **persistent** client so the index survives restarts, cosine distance (`hnsw:space = cosine`),
and `upsert` so re-running the notebook does not duplicate chunks.

In [7]:
import chromadb

chroma_client = chromadb.PersistentClient(path=str(DB_DIR))
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

BATCH = 256
for i in range(0, len(chunks), BATCH):
    batch = chunks[i:i + BATCH]
    collection.upsert(
        ids=[c["id"] for c in batch],
        documents=[c["text"] for c in batch],
        metadatas=[c["metadata"] for c in batch],
        embeddings=chunk_embeddings[i:i + BATCH].tolist(),
    )

print("Vectors in collection:", collection.count())

Vectors in collection: 405


## 6. Retrieve

The question is embedded with the same model and the `TOP_K` nearest chunks are returned.
ChromaDB returns a cosine **distance** (0 = identical); we convert it to a similarity score for readability.

In [8]:
def retrieve(question: str, k: int = TOP_K) -> list[dict]:
    q_emb = embed_texts([question])[0].tolist()
    res = collection.query(query_embeddings=[q_emb], n_results=k,
                           include=["documents", "metadatas", "distances"])
    hits = []
    for text, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({"text": text, "metadata": meta, "score": 1 - dist})
    return hits

def show_hits(hits: list[dict], width: int = 110, preview: int = 300):
    for i, h in enumerate(hits, 1):
        m = h["metadata"]
        print(f"[{i}] score={h['score']:.3f}  {m['source']}  >  {m['section']}")
        print(textwrap.indent(textwrap.fill(h['text'][:preview].replace('\n', ' '), width), "    "), "...\n")

show_hits(retrieve("How does the discriminator in a GAN learn?"))

[1] score=0.755  generative_adversarial_network.md  >  Generative adversarial network
    trained to minimize the distance to a specific image, but rather to fool the discriminator. This enables the
    model to learn in an unsupervised manner. GANs are similar to mimicry in evolutionary biology, with an
    evolutionary arms race between both networks. ...

[2] score=0.722  generative_adversarial_network.md  >  Mathematical
    GANs are similar to mimicry in evolutionary biology, with an evolutionary arms race between both networks. The
    original GAN is defined as the following game:  > Each probability space(Ω,μref) defines a GAN game. > > There
    are 2 players: generator and discriminator. > > The generator's strategy set i ...

[3] score=0.700  generative_adversarial_network.md  >  Generative adversarial network
    al in June 2014. In a GAN, two neural networks compete with each other in the form of a zero-sum game, where
    one agent's gain is another agent's loss. Given a 

## 7. Generate with Claude

The retrieved chunks are numbered and placed in the prompt as *context*. The system prompt instructs the model to:

- answer **only** from the context,
- cite the chunks it used as `[1]`, `[2]`, …,
- say explicitly when the context does not contain the answer (this is what reduces hallucinations).

In [9]:
import anthropic

client = anthropic.Anthropic()  # key comes from ANTHROPIC_API_KEY

SYSTEM_PROMPT = '''You are a teaching assistant for a Generative AI course.
Answer the user's question using ONLY the numbered context passages provided.
Rules:
- Cite the passages you rely on with bracketed numbers, e.g. [2] or [1][3].
- If the context does not contain the answer, reply exactly: "I could not find this in the provided documents." and nothing else.
- Be concise and factual: 2-6 sentences, or a short list if the question asks for several items.
- Do not use knowledge that is not in the context, even if you know the answer.'''

def build_prompt(question: str, hits: list[dict]) -> str:
    context = "\n\n".join(
        f"[{i}] (source: {h['metadata']['source']}, section: {h['metadata']['section']})\n{h['text']}"
        for i, h in enumerate(hits, 1)
    )
    return f"Context passages:\n\n{context}\n\nQuestion: {question}"

def generate(question: str, hits: list[dict], max_tokens: int = 1024) -> str:
    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=max_tokens,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": build_prompt(question, hits)}],
    )
    if response.stop_reason == "refusal":
        return f"[The model declined to answer: {response.stop_details and response.stop_details.explanation}]"
    return "".join(block.text for block in response.content if block.type == "text").strip()

def ask(question: str, k: int = TOP_K, show_sources: bool = True) -> str:
    hits = retrieve(question, k)
    answer = generate(question, hits)
    print("Q:", question)
    print("A:", textwrap.fill(answer, 110, subsequent_indent="   "))
    if show_sources:
        print("\nSources:")
        for i, h in enumerate(hits, 1):
            m = h["metadata"]
            print(f"  [{i}] {m['source']} > {m['section']}  (score {h['score']:.3f})")
    print("-" * 110)
    return answer

_ = ask("What is the difference between the generator and the discriminator in a GAN?")

Q: What is the difference between the generator and the discriminator in a GAN?
A: In a GAN, the generator and discriminator are two opposing players in a zero-sum game [1]. They differ in what
   they produce and in their goals:  - **What they produce:** The generator's strategy is a probability
   distribution μG over the data space Ω. It generates samples meant to resemble the reference data [1]. The
   discriminator maps each input x to a probability on [0,1], which judges whether the input comes from the
   real distribution [1]. In practice, the discriminator is usually a deterministic function D: Ω→[0,1],
   typically a deep neural network [4]. - **How they are implemented:** The generator is usually a function G
   that transforms random noise z, drawn from an easy distribution such as a uniform or Gaussian, into samples
   G(z) [4]. - **Their goals:** Under the objective L(G,D) = E[ln D(x)] + E[ln(1−D(G(z)))], the generator
   tries to minimize the objective and the discrimina

## 8. Evaluation

### 8.1 Retrieval quality

For each test question we know which document should contain the answer. **Hit@k** = share of questions
for which the expected document appears among the top-k retrieved chunks. This checks the retriever on its own,
independently of the LLM.

In [10]:
TEST_QUESTIONS = [
    # (question, expected source document)
    ("What are the main components of the original Transformer encoder layer?", "transformer.md"),
    ("What is the scaled dot-product attention formula and why is it scaled?", "attention_mechanism.md"),
    ("How are large language models trained, and what is the role of pre-training and fine-tuning?", "large_language_model.md"),
    ("What are the forward and reverse processes in a diffusion model?", "diffusion_model.md"),
    ("Why is mode collapse a problem when training GANs?", "generative_adversarial_network.md"),
    ("What problems of LLMs does retrieval-augmented generation address?", "retrieval_augmented_generation.md"),
    ("What is a hallucination in the context of language models?", "large_language_model.md"),
    ("What is the purpose of positional encoding in a Transformer?", "transformer.md"),
]

def hit_at_k(k: int) -> float:
    hits_count = 0
    for q, expected in TEST_QUESTIONS:
        sources = {h["metadata"]["source"] for h in retrieve(q, k)}
        hits_count += expected in sources
    return hits_count / len(TEST_QUESTIONS)

for k in (1, 3, 5):
    print(f"Hit@{k}: {hit_at_k(k):.0%}")

print("\nTop-1 source per question:")
for q, expected in TEST_QUESTIONS:
    top = retrieve(q, 1)[0]["metadata"]["source"]
    mark = "OK " if top == expected else "MISS"
    print(f"  {mark}  {top:38}  {q[:70]}")

Hit@1: 100%


Hit@3: 100%


Hit@5: 100%

Top-1 source per question:
  OK   transformer.md                          What are the main components of the original Transformer encoder layer
  OK   attention_mechanism.md                  What is the scaled dot-product attention formula and why is it scaled?
  OK   large_language_model.md                 How are large language models trained, and what is the role of pre-tra
  OK   diffusion_model.md                      What are the forward and reverse processes in a diffusion model?
  OK   generative_adversarial_network.md       Why is mode collapse a problem when training GANs?
  OK   retrieval_augmented_generation.md       What problems of LLMs does retrieval-augmented generation address?
  OK   large_language_model.md                 What is a hallucination in the context of language models?
  OK   transformer.md                          What is the purpose of positional encoding in a Transformer?


### 8.2 End-to-end answers

Run the whole pipeline on the test questions plus one question that is **not** covered by the corpus, to verify that the model refuses instead of inventing an answer.

In [11]:
for q, _ in TEST_QUESTIONS[:4]:
    ask(q)

# a question the documents cannot answer -> the model should say it cannot find it
ask("Who won the FIFA World Cup in 2018?")

Q: What are the main components of the original Transformer encoder layer?
A: In the original Transformer, each encoder layer has two sublayers: a **self-attention** sublayer and a
   **feedforward network** [2].  - **Self-attention** builds contextualized representations, so each token's
   representation "mixes" in information from the other input tokens [4]. - **Feedforward network** further
   processes the outputs. These feed-forward layers hold most of the model's parameters [4]. - **Residual
   connections and layer normalization** wrap the sublayers. They are needed for numerical stability and
   convergence [2][4].  The original 2017 transformer placed layer normalization after each sublayer (the
   post-LN convention). Many later diagrams instead use pre-LN [1].

Sources:
  [1] transformer.md > Transformer  (score 0.676)
  [2] transformer.md > Sublayers  (score 0.648)
  [3] transformer.md > Architecture  (score 0.648)
  [4] transformer.md > Encoder–decoder (overview)  (score 

Q: What is the scaled dot-product attention formula and why is it scaled?
A: The scaled dot-product (QKV) attention formula is **Attention(Q, K, V) = softmax(QKᵀ / dₖ) V**, as rendered in
   the passages, whose formatting appears to have lost part of the scaling notation [1][3].  - **Q** (m×dₖ)
   holds m queries, and **K** (n×dₖ) and **V** (n×dᵥ) hold an unordered set of n key-value pairs [1]. - In a
   transformer, these matrices come from learned weights: Q = X_query·W_Q, K = X_key·W_K, V = X_value·W_V [2].
   - The softmax is applied to each row independently. Its weights are used to sum the value vectors, so each
   output row is a weighted combination of V's rows [1][3]. - The query and key dimensions must match (d_query
   = d_key), and the key and value sequences must have the same length [3].  The provided documents do not
   explain why the dot products are scaled by the key dimension dₖ.

Sources:
  [1] attention_mechanism.md > Standard scaled dot-product attention  (score 0

Q: How are large language models trained, and what is the role of pre-training and fine-tuning?
A: LLMs (like GPT models) are typically trained in two stages: **pre-training** followed by **fine-tuning**
   [2][4].  - **Pre-training:** The model is first trained to predict the next word on a large amount of data
   [2]. This is done through self-supervised learning on a large, generic, typically unlabeled corpus such as
   The Pile [4]. As a result, most LLMs are essentially next-token predictors before they are fine-tuned [1].
   - **Fine-tuning:** This stage shapes the model's behavior [1]. In the general transformer approach, it
   means supervised fine-tuning on a small, task-specific dataset [4]. For LLMs, specific techniques include:
   - **Instruction fine-tuning**, a form of supervised learning that teaches models to follow user
   instructions (e.g., OpenAI's InstructGPT in 2022) [1].   - **Reinforcement learning from human feedback
   (RLHF)**, in which a reward model is trai

Q: What are the forward and reverse processes in a diffusion model?
A: A diffusion model has two main components: the **forward diffusion process** and the **reverse sampling
   process** [3].  - **Forward process:** Noise is added to the data, progressively converting an unknown data
   distribution (e.g., natural-looking images) into a known one, a standard Gaussian [1][2]. It is an SDE that
   can be integrated in closed form, so it costs nothing to compute [1]. In continuous time it takes the form
   dx_t = −½β(t)x_t dt + √β(t) dW_t, where W_t is a Wiener process. This is a special case of the overdamped
   Langevin equation, which is why "Langevin dynamics" is sometimes mentioned [4]. - **Reverse (backward)
   process:** This process removes noise. The model learns the score function and then uses it to run the
   diffusion backward [4]. It is also an SDE, but it cannot be integrated in closed form, so it must be solved
   step by step with standard SDE solvers, which can be very 

Q: Who won the FIFA World Cup in 2018?
A: I could not find this in the provided documents.

Sources:
  [1] generative_adversarial_network.md > History  (score 0.150)
  [2] generative_adversarial_network.md > Invertible data augmentation  (score 0.148)
  [3] generative_adversarial_network.md > Fashion, art and advertising  (score 0.148)
  [4] generative_adversarial_network.md > Fashion, art and advertising  (score 0.125)
  [5] generative_adversarial_network.md > InfoGAN  (score 0.119)
--------------------------------------------------------------------------------------------------------------


'I could not find this in the provided documents.'

### 8.3 RAG vs. no-RAG

The same question asked **without** retrieved context. Compare the grounding and the citations: without RAG the model
answers from its parametric memory (possibly outdated or imprecise, and unverifiable); with RAG every claim points to a passage.

In [12]:
def generate_without_rag(question: str, max_tokens: int = 1024) -> str:
    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=max_tokens,
        system="You are a teaching assistant for a Generative AI course. Answer concisely in 2-6 sentences.",
        messages=[{"role": "user", "content": question}],
    )
    return "".join(b.text for b in response.content if b.type == "text").strip()

question = "What problems of LLMs does retrieval-augmented generation address?"

print("WITHOUT RAG:")
print(textwrap.fill(generate_without_rag(question), 110), "\n")
print("WITH RAG:")
_ = ask(question)

WITHOUT RAG:


Retrieval-augmented generation (RAG) addresses several key LLM limitations by grounding responses in external
documents fetched at query time:  - **Knowledge cutoff:** It gives the model up-to-date information beyond its
training data. - **Hallucination:** Anchoring answers in retrieved evidence reduces fabricated facts. -
**Missing domain or private knowledge:** It lets the model use proprietary or specialized data without costly
retraining or fine-tuning. - **Lack of verifiability:** Responses can cite their sources, which improves
transparency and trust.  RAG also makes knowledge easier to update, since you only change the document store,
not the model weights. 

WITH RAG:


Q: What problems of LLMs does retrieval-augmented generation address?
A: Retrieval-augmented generation (RAG) addresses several limitations of LLMs:  - **Outdated or missing
   knowledge:** LLMs are limited to their training data. RAG lets them use domain-specific or updated
   information from external sources, such as internal company data or authoritative sources [5]. - **Costly
   retraining:** When new information becomes available, you can update the external knowledge base instead of
   retraining the model, which saves computational and financial costs [1][3]. - **Fabricated information:**
   The context points to problems like chatbots describing policies that don't exist or recommending
   nonexistent legal cases to lawyers, which RAG aims to mitigate [3]. - **Lack of transparency:** RAG lets
   LLMs include sources in their responses, so users can cross-check the retrieved content for accuracy and
   relevance [3].

Sources:
  [1] retrieval_augmented_generation.md > Process 

## 9. Try your own question

Edit the question below and run the cell.

In [13]:
_ = ask("How does a diffusion model generate an image from noise?")

Q: How does a diffusion model generate an image from noise?
A: A diffusion model is trained to reverse the process of adding Gaussian noise to an image. After training, it
   generates an image by starting from pure random noise and repeatedly applying the network to denoise it
   [2]. In DDPM, the network takes a time step t and a noisy image xt and predicts the noise ϵθ(xt,t).
   Predicting the noise is equivalent to predicting the denoised image, because the noise can be subtracted
   from xt [5]. This network, called the "backbone," is typically a U-Net or a transformer [2][5].  More
   abstractly, the model learns a probability path that connects the distribution of natural-looking images to
   a standard Gaussian distribution. This path is defined implicitly by the score function ∇ln pt [4].
   Generation follows the path in reverse, from noise back to an image.  For text-conditioned generation,
   systems like Stable Diffusion and DALL-E combine the diffusion model with text enc

## 10. Conclusions

- **Corpus**: 6 Wikipedia articles (~38k words) were loaded, split into structure-aware overlapping chunks and embedded with `all-MiniLM-L6-v2`.
- **Vector store**: ChromaDB with cosine similarity, persisted on disk, so indexing runs once.
- **Retrieval**: top-k nearest chunks; Hit@k on the test set shows how often the right document is found.
- **Generation**: Claude answers only from the retrieved passages and cites them; for out-of-corpus questions it declines, which demonstrates how RAG reduces hallucinations compared with the no-RAG baseline.

Possible improvements: hybrid search (BM25 + embeddings), a cross-encoder re-ranker, larger embedding model,
query rewriting, and a proper answer-quality evaluation (e.g. LLM-as-judge against reference answers).